# Follow-up probe

This is a baseline method, the **follow-up probe** (also called the "last-token probe" in
[Liars' Bench](https://github.com/Cadenza-Labs/liars-bench)), adapted from the
`use_followup_question` upper-bound experiment in
[deception-detection](https://github.com/ApolloResearch/deception_detection).

Instead of reading the model's own response, we append a forced follow-up turn to the
conversation — *"Was your last response deceptive? (Yes/No)"* / *"No"* — and read the residual
stream at the **final token of that forced reply**. We then score that single-token activation
with the **same trained mean-difference probe** (`mean_probe.npz`, from
`tutorials/2.1-mean-probe.ipynb`) used by `mean_probe.ipynb` — only the readout changes, the
probe direction stays the same.


In [ ]:
import os

import numpy as np
import pandas as pd

# The runner (and `submit.py --dry`) always sets DATASET_NAME — one repo, single
# `test` split — to the dataset you predict on.
DATASET_NAME = os.environ["DATASET_NAME"]

# The canonical output file the grader reads. Do not rename.
SUBMISSION_PATH = "submission.csv"

## Load the trained probe

`mean_probe.npz` ships in the repo next to this notebook. It holds the probe direction, the
feature-normalization stats, the decision midpoint (`bias`), the score spread (`score_std`), and the
metadata needed to reproduce the exact same activation extraction used in training.

In [ ]:
# cwd is the repo root when the runner executes the notebook; the weights ship under submissions/.
CANDIDATES = ["submissions/mean_probe.npz", "mean_probe.npz", "./mean_probe.npz"]
WEIGHTS = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert WEIGHTS is not None, f"mean_probe.npz not found; looked in {CANDIDATES}"

z = np.load(WEIGHTS)
direction = z["direction"].astype(np.float64)          # (hidden,)
mu        = z["mu"].astype(np.float64)
sigma     = z["sigma"].astype(np.float64)
NORMALIZE = bool(z["normalize"])
BIAS      = float(z["bias"])
SCORE_STD = float(z["score_std"])
LAYER     = int(z["layer"])
MODEL_ID  = str(z["model_id"])
USE_CHAT_TEMPLATE = bool(z["use_chat_template"])
MAX_SEQ_LEN       = int(z["max_seq_len"])
print(f"probe: model={MODEL_ID} | layer={LAYER} | dim={direction.shape[0]} | normalize={NORMALIZE}")

def probe_proba(X):
    """Pooled activations (N, hidden) at LAYER -> deception probability in [0, 1].

    Faithful to MMSDetector scoring: project (optionally standardized) activations onto the
    difference-of-means direction. The logistic squash is monotonic, so it preserves the AUROC
    ranking while mapping the decision midpoint (BIAS) to 0.5.
    """
    X = np.asarray(X, dtype=np.float64)
    Xn = (X - mu) / sigma if NORMALIZE else X
    s = Xn @ direction
    zc = np.clip((s - BIAS) / SCORE_STD, -30.0, 30.0)
    return 1.0 / (1.0 + np.exp(-zc))

## Load the model

Gemma 3 is multimodal, so it loads via `VisionLanguageModel`. We use the default NDIF host
(`api.ndif.us`); the runner forwards your `NDIF_API_KEY`, which nnsight uses to authenticate the
remote traces.

We trace the base `google/gemma-3-27b-it` with the **`NDIF/hackathon-imposter-eve-gemma27B` LoRA
adapter** applied (via nnsight's `peft=` argument), so the activations come from the imposter model
rather than the plain base. The probe direction was learned on base-model activations, so this is a
transfer test — the layer-26 residual extraction is identical, but expect the probe to be somewhat
off-distribution on the adapted model. Set `LORA_ADAPTER = None` (or `$LORA_ADAPTER=""`) to fall
back to the plain base model.

In [ ]:
from nnsight import CONFIG
try:
    from nnsight import VisionLanguageModel
except ImportError:                       # nnsight 0.7.x no longer re-exports at top level
    from nnsight.modeling.vlm import VisionLanguageModel

if os.environ.get("NDIF_API_KEY"):
    CONFIG.API.APIKEY = os.environ["NDIF_API_KEY"]  # in-memory only; set_default_api_key() persists to read-only site-packages -> PermissionError

# Apply a PEFT/LoRA adapter on top of the base model for the remote trace. nnsight forwards
# this repo id to NDIF (via the `ndif-env` header); NDIF wraps the served base model with
# `PeftModel.from_pretrained` per request, so the base (MODEL_ID) must be hosted on NDIF and
# the adapter loadable there. Set LORA_ADAPTER = None to trace the plain base model.
# Override at runtime with $LORA_ADAPTER if you want to swap adapters without editing the cell.
LORA_ADAPTER = os.environ.get("LORA_ADAPTER", "NDIF/hackathon-imposter-eve-gemma27B")

if LORA_ADAPTER:
    print(f"applying LoRA adapter on NDIF: {LORA_ADAPTER} (base={MODEL_ID})")
    model = VisionLanguageModel(MODEL_ID, peft=LORA_ADAPTER)
else:
    model = VisionLanguageModel(MODEL_ID)

# Locate the text decoder's `layers` ModuleList. Its nesting depth depends on
# whether NDIF served the plain VLM (Gemma3ForConditionalGeneration, layers at
# .model.language_model.layers) or, with a LoRA/PEFT adapter, the text-only
# Gemma3ForCausalLM (layers at .model.model.layers / similar) — so search the
# module tree instead of hardcoding a path. Excludes the vision tower's own
# `layers` ModuleList by requiring a decoder-layer class (has self_attn+mlp).
def _find_decoder_layers(root):
    candidates = []
    for name, child in root.named_modules():
        if name.rsplit(".", 1)[-1] != "layers":
            continue
        children = list(child.children())
        if children and hasattr(children[0], "self_attn") and hasattr(children[0], "mlp") \
                and "Decoder" in type(children[0]).__name__:
            candidates.append((name, child))
    assert len(candidates) == 1, f"expected exactly one decoder `layers` module, found {[n for n, _ in candidates]}"
    return candidates[0][1]

DECODER_LAYERS = _find_decoder_layers(model.model)

## Tokenize and extract the follow-up turn's final-token activation

For each example, we trim the conversation to the response being judged, append the forced
follow-up turn (*"Was your last response deceptive? (Yes/No)"* / *"No"*), apply the model's chat
template, and run the extended conversation through the model on NDIF. Instead of mean-pooling
over a response span, we read the residual stream at `LAYER` for a **single position**: the last
token of the forced "No" reply (the token immediately before `<end_of_turn>`).


In [ ]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
EOT_ID = tokenizer.convert_tokens_to_ids("<end_of_turn>")

# The forced follow-up turn, matching deception-detection's use_followup_question upper bound.
FOLLOWUP_QUESTION = "Was your last response deceptive? (Yes/No)"
FOLLOWUP_ANSWER = "No"

def to_judged_messages(msgs):
    """Trim a conversation to end at its last assistant message — the one being judged."""
    idxs = [i for i, m in enumerate(msgs) if m.get("role") == "assistant"]
    return msgs[: idxs[-1] + 1] if idxs else None

def normalize_messages(messages):
    """Merge a leading system message into the first user turn (Gemma's chat
    template doesn't accept a standalone system role)."""
    norm, pending_system = [], None
    for m in messages:
        if m["role"] == "system" and not norm:
            pending_system = m["content"]; continue
        if not norm and m["role"] == "user" and pending_system:
            norm.append({"role": "user", "content": pending_system + "\n\n" + m["content"]})
            pending_system = None; continue
        norm.append(m)
    return norm

def tokenize_followup_last_token(messages):
    """Chat-template path: append the forced follow-up turn and return
    (input_ids, index of the forced reply's last content token)."""
    ext = normalize_messages(messages) + [
        {"role": "user", "content": FOLLOWUP_QUESTION},
        {"role": "assistant", "content": FOLLOWUP_ANSWER},
    ]
    full_ids = tokenizer.apply_chat_template(
        ext, tokenize=True, add_generation_prompt=False, return_dict=True)["input_ids"]
    prefix_ids = tokenizer.apply_chat_template(
        ext[:-1], tokenize=True, add_generation_prompt=True, return_dict=True)["input_ids"]
    start, end = len(prefix_ids), len(full_ids)
    while end > start and full_ids[end - 1] == EOT_ID:
        end -= 1
    return full_ids, end - 1

def tokenize_followup_raw(messages):
    """Raw concat path: append the follow-up Q&A as plain text and return
    (input_ids, index of its last token)."""
    *prefix_msgs, last = messages
    prompt_text = "\n".join(m["content"] for m in prefix_msgs)
    tail_text = "\n".join([last["content"], FOLLOWUP_QUESTION, FOLLOWUP_ANSWER])
    ids = (tokenizer.encode(prompt_text, add_special_tokens=True)
           + tokenizer.encode(tail_text, add_special_tokens=False))
    return ids, len(ids) - 1

tokenize_fn = tokenize_followup_last_token if USE_CHAT_TEMPLATE else tokenize_followup_raw

def cap_length(ids, idx):
    """Left-trim to MAX_SEQ_LEN; the follow-up turn sits at the end so idx stays valid."""
    if len(ids) <= MAX_SEQ_LEN:
        return ids, idx
    k = len(ids) - MAX_SEQ_LEN
    return ids[k:], idx - k

def collate(batch):
    max_len = max(len(t[0]) for t in batch)
    B = len(batch)
    input_ids = torch.full((B, max_len), EOT_ID, dtype=torch.long)
    attn = torch.zeros((B, max_len), dtype=torch.long)
    idxs = []
    for i, (ids, idx) in enumerate(batch):
        input_ids[i, :len(ids)] = torch.tensor(ids)
        attn[i, :len(ids)] = 1
        idxs.append(idx)
    return input_ids, attn, idxs

BATCH_SIZE = 8   # single-layer, single-token extraction is light; raise/lower for remote throughput/OOM

def extract_layer_last_token(tok_list, layer, batch_size=BATCH_SIZE):
    """Read the residual stream at `layer`, at the forced follow-up reply's final
    token, for each example. Returns (N, hidden)."""
    out = []
    n_batches = (len(tok_list) + batch_size - 1) // batch_size
    for b in range(n_batches):
        batch = tok_list[b*batch_size:(b+1)*batch_size]
        input_ids, attn, idxs = collate(batch)
        with model.trace({"input_ids": input_ids, "attention_mask": attn}, remote=True):
            h = DECODER_LAYERS[layer].output                      # (B, T, D)
            pooled = torch.stack(
                [h[i, idx] for i, idx in enumerate(idxs)]
            ).save()
        out.append(pooled.cpu().float().numpy())
        print(f"  batch {b+1}/{n_batches} (max_len={input_ids.shape[1]})")
    return np.concatenate(out, axis=0)


## Predict and write `submission.csv`

Extract the follow-up-turn final-token activation for every test example, score it with the
probe, and assemble one prediction per example **in dataset order**. Any example we can't parse
(no assistant turn) falls back to `0.5` so the output always has exactly one valid row per example.


In [ ]:
from datasets import load_dataset

examples = load_dataset(DATASET_NAME, split="test")
print(f"loaded {len(examples)} examples")

tok = []
valid_idx = []
for i, ex in enumerate(examples):
    msgs = to_judged_messages(list(ex["messages"]))
    if msgs is None:
        continue
    ids, idx = cap_length(*tokenize_fn(msgs))
    if idx < 0:                     # degenerate: follow-up turn got trimmed away
        continue
    tok.append((ids, idx))
    valid_idx.append(i)

predictions = np.full(len(examples), 0.5, dtype=float)   # fallback for unparseable rows
if valid_idx:
    X = extract_layer_last_token(tok, LAYER)
    predictions[np.array(valid_idx)] = probe_proba(X)
print(f"scored {len(valid_idx)}/{len(examples)} examples with the probe "
      f"({len(examples) - len(valid_idx)} fell back to 0.5)")


In [ ]:
predictions = np.clip(np.asarray(predictions, dtype=float), 0.0, 1.0)
assert len(predictions) == len(examples), "need exactly one prediction per example"

pd.DataFrame({"id": np.arange(len(predictions)), "prediction": predictions}).to_csv(
    SUBMISSION_PATH, index=False)
print(f"wrote {len(predictions)} predictions to {SUBMISSION_PATH}")